# H&M Product Recommendation Engine
**Author:** Karim Mattar | DS207 Applied Machine Learning, Spring 2026

## Overview
2-stage recommendation system:
1. **KNN Retrieval** — narrows 105k products to ~100 candidates per customer using cosine similarity on product feature vectors
2. **Ranker** — scores each candidate with a Neural Net and a Random Forest, picks top 12

**Evaluation:** Precision@12, Recall@12, F1@12 (per customer, then averaged)

In [ ]:
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras

sys.path.insert(0, '../src')
from customer_features import CustomerFeatureEngineer
from product_features import ProductFeatureEngineer
from recommendation_training import RecommendationTrainingBuilder

warnings.filterwarnings('ignore')
np.random.seed(67)
tf.random.set_seed(67)

In [ ]:
# Raw cleaned CSVs
DATA_DIR = Path('/Users/karimmattar11/Desktop/Berkeley/ds207/Project/archive (4)')

articles = pd.read_csv(DATA_DIR / 'articles_hm_cleaned.csv')
customers = pd.read_csv(DATA_DIR / 'customer_hm_cleaned.csv')
transactions = pd.read_csv(DATA_DIR / 'transactions_hm_cleaned.csv')
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

print(f"Articles: {articles.shape}")
print(f"Customers: {customers.shape}")
print(f"Transactions: {transactions.shape}")
print(f"Date range: {transactions['t_dat'].min()} → {transactions['t_dat'].max()}")

In [ ]:
PKL_DIR = Path('../data/processed/product_recommendation')

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

X_train = load_pkl(PKL_DIR / 'X_train_base.pkl')
y_train = load_pkl(PKL_DIR / 'y_train_base.pkl')
X_val   = load_pkl(PKL_DIR / 'X_val_base.pkl')
y_val   = load_pkl(PKL_DIR / 'y_val_base.pkl')
X_test  = load_pkl(PKL_DIR / 'X_test_base.pkl')
y_test  = load_pkl(PKL_DIR / 'y_test_base.pkl')

print(f"Train: {X_train.shape}, positives: {y_train.sum()}")
print(f"Val:   {X_val.shape},   positives: {y_val.sum()}")
print(f"Test:  {X_test.shape},  positives: {y_test.sum()}")
print(f"Features ({X_train.shape[1]}): {X_train.columns.tolist()}")